# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [2]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [3]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-DsHn7OlhlxW5rytB9WQyoqSZiJqUS',
 'object': 'chat.completion',
 'created': 1781830473,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Fun fact: Honey never spoils. Archaeologists have found edible honey in ancient Egyptian tombs that are over 3,000 years old! Want another fun fact, maybe about space or animals?',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 11,
  'completion_tokens': 497,
  'total_tokens': 508,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 448,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [4]:
response.json()["choices"][0]["message"]["content"]

'Fun fact: Honey never spoils. Archaeologists have found edible honey in ancient Egyptian tombs that are over 3,000 years old! Want another fun fact, maybe about space or animals?'

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [2]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me another fun fact"}])

response.choices[0].message.content



"Here's a fun one: Venus spins extremely slowly and in the opposite direction from most planets. It takes about 243 Earth days to rotate once (a Venusian day), which is longer than its year—about 225 Earth days. Want another fun fact?"

## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

## THIS IS OPTIONAL - but if you wish to try out Google Gemini, please visit:

https://aistudio.google.com/

And set up your API key at

https://aistudio.google.com/api-keys

And then add your key to the `.env` file, being sure to Save the .env file after you change it:

`GOOGLE_API_KEY=AIz...`


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [8]:
import requests
requests.get("http://localhost:11434").content

b'Ollama is running'

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [6]:
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 534 KB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 4.2 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  17 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  24 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  28 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   3% ▕                  ▏  55 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   3% ▕                  ▏  67 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   3% ▕                  ▏  68 MB/2.0 GB                  pulling

In [9]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [10]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

"Did you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and are still edible! Honey's unique composition, which includes hydrogen peroxide, an antibacterial agent, makes it virtually untouchable by microorganisms. That's the power of honey!"

In [11]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 573 KB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏  10 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   2% ▕                  ▏  19 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   3% ▕                  ▏  32 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   3% ▕                  ▏  38 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   4% ▕                  ▏  47 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   5% ▕                  ▏  52 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   6% ▕█                 ▏  63 MB/1.1 GB                  pulling manifest 
pulling 

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [12]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'\n\nEven though a car reaches top speeds of up to 60 miles per hour on highways, in real traffic, it slows down significantly for safety reasons. Each vehicle moves at a lower speed than its highest possible speed due to spacing issues, which reduces the overall flow of cars on the road.'

In [15]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [16]:
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [18]:
import requests
requests.get("http://localhost:11434").content

b'Ollama is running'

In [20]:
from openai import OpenAI

openai = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

response = openai.chat.completions.create(
    model="llama3.2",
    messages=messages
)

print(response.choices[0].message.content)

assistant>

# The Tea is Spilled
## Website Summary

*   *Hyping up their services*: "Our team of expert technicians will fix your phone/laptop/computer issue within 24-48 hours... or forever, we're not really sure what the deadline is."
*   *Product comparisons gone wild*: Compare the features of 5 different phones side by side to see which one has more RAM. Spoiler alert: They all have about 4 GB.
*   *Marketing jargon*: "Innovative" and "groundbreaking" marketing campaigns that will make you doubt your own perception.

# The Verdict 

This site is either trying to entertain us or drive us crazy, we're not quite sure which one.


In [24]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI


In [25]:
ed = fetch_website_contents("https://edwarddonner.com")
ed

'Home - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula.io\n. I was previously founder and CEO of AI startup untapt,\nacquired in 2021\n, and a Managing Director at JPMorgan.\nI will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 600,000 enrolled across 194 countries. The\nfull curriculum is here\n. If you’re visiting from one of my courses –

In [ ]:
messages = [
    {
        "role": "system",
        "content": """You are a snarky assistant that analyzes the contents of a website and provides a short, snarky, humorous summary, ignoring text that might be navigation related.

Respond in markdown. Do not wrap the markdown in a code block; respond only with markdown."""
    }
]
{"role": "user", "content": """
Here are the contents of a website. https://www.nippo.in
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""}

response = openai.chat.completions.create(model="llama3.2", messages=messages)
response.choices[0].message.content

'assistant>\n\n# Okay, Let\'s Get This Over With\n\n## The "About Us" Page (Yawn)\n\n* Apparently, [Company Name] is trying to justify its existence by listing all the reasons why it\'s cool and stuff.\n* Founded 20 years ago, but somehow still relevant apparently?\n* Team: A mix of clichés including "dedicated", "passionate", and "visionary" - because that\'s not overused.\n\n## The Values Page (Boring)\n\n* Another thrilling read about being "committed to excellence", "driven by innovation", and "focused on customer satisfaction".\n* What sets you apart from every other company?\n\t+ Oh wait, it\'s free coffee Fridays.\n* At least they\'re consistent in being unoriginal.\n\n## The Blog (Somewhere Interesting This Time)\n\n* [Category] articles that are written by experts, informed by data, and offer nuanced insights into the world of [Industry].\n* Bonus points if you manage to not induce a coma with corporate jargon.\n* Will keep reading for now...'

In [29]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [30]:
messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula

In [31]:
def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [32]:
summarize("https://www.nippo.in")

"### Nippo's 50+ Year Anniversary and Brand Rebranding Efforts\n\n*   Nippo, an Indian company, is celebrating its 50+ years of empowering millions with innovative products, including batteries, torches, LED lighting, electrical accessories, and mosquito protection.\n*   The company boasts a fresh outlook on innovation and aims to shape a brighter future through their diverse product offerings.\n\n### Guiding Values\n\nNippo's guiding values include:\n\n1.  Fairness: Treating all stakeholders with fairness and honesty in dealings.\n2.  Integrity: Honesty and truthfulness in business transactions.\n3.  Excellence: Striving for high standards and exceeding expectations.\n4.  Empowerment: Taking accountability for individual and collective actions.\n5.  Differential Thinking: Developing unique ideas to improve customer experience and business results"

In [33]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [34]:
display_summary("https://www.nippo.in")

A household name has been around for more than 50 years, and they're still going strong. They sell batteries, torches, lighting, and other stuff that's useful (obviously).

As for news announcements with the same level of excitement, there isn't anything groundbreaking here, but you can find some stuff about following them on social media... because, priorities.

In [35]:
display_summary("https://www.apple.com/apple-music/")

**Apple Music: Because You Clearly Need More Music in Your Life**

Apple Music promises "the best music experience ever," with over 100 million songs, ad-free listening, and immersive Spatial Audio. Oh, and they'll throw in some exclusive content and live concerts for good measure. But let's be real, you're probably just going to get lost in the sea of playlists and discover new artists through their algorithmic magic.

**New Music Highlights:**

* **Kendrick Lamar**: Because who doesn't love some good Kendrick?
* **Spatial Audio with Dolby Atmos**: The latest sound tech that'll make your music-listening experience feel truly epic.
* **Music Discovery**: Apple's got a playlist for every mood and taste, because why choose when you can have it all?